# PyTorch training

Companion notebook for the RDDAC [documentation](https://rddac.readthedocs.io). It is written to stand on its own: every step is annotated so the notebook reads top to bottom.

## Walkthrough

1. construct an `RDDACDataset` over a published view,
2. stream single records through a `DataLoader` (the raw signals are ragged),
3. batch properly with a fixed-shape custom view via `add_view` + `dataset=`,
4. filter via process-parameter columns and an explicit `sim_ids` allowlist,
5. build the canonical train / val / test splits from the CSV `split` column,
6. enable per-shard shuffle and `set_epoch` for multi-epoch training,
7. understand the metadata-column limitation and its workarounds,
8. run a minimal training-loop skeleton.

## Assumptions

- This notebook lives in `notebooks/` of the repository and reads the dataset from `../data/` (on your machine: the directory `rddac download` wrote to).
- The data directory contains `metadata.json`, `process_parameters.csv`, and at least the bundled `sample.zip`.
- The PyTorch extra is installed: `pip install rddac[torch]`.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import zipfile
from pathlib import Path

import rddac
from rddac import RDDACDataset
from torch.utils.data import DataLoader

print('rddac', rddac.__version__)

DATA_DIR = Path('../data')
# DATA_DIR = Path('./data')   # uncomment instead when running from the repository root
assert (DATA_DIR / 'metadata.json').is_file(), f'{DATA_DIR} does not contain the dataset'

# The 18 experiments bundled in sample.zip (one per category) - a cheap allowlist for the demos below.
with zipfile.ZipFile(DATA_DIR / 'h5' / 'sample.zip') as zf:
    SAMPLE_IDS = sorted(int(Path(n).stem) for n in zf.namelist() if n.endswith('.h5'))
print('sample experiments:', SAMPLE_IDS)

rddac 0.1.0
sample experiments: [0, 500, 1002, 1500, 2000, 2500, 3000, 3500, 4000, 4500, 5000, 5500, 6000, 6500, 7000, 7500, 8000, 8500]


## 1. Construct an `RDDACDataset`

`RDDACDataset(view, data_dir, ...)` is a `torch.utils.data.IterableDataset` that streams records of a Croissant view. At construction time it:

1. loads the manifest (same code path as `rddac.load`),
2. resolves the view's fields to HDF5 paths and JSONPath transforms,
3. scans `data_dir` for zip files and builds an index `experiment id -> local zip path`,
4. reads `process_parameters.csv` and applies the optional `sim_ids` allowlist and `where` predicate.

(The `sim_ids` argument name is kept for drop-in DDACS compatibility — for RDDAC the ids are experiment ids.) After construction it knows exactly which experiments are locally available; iteration silently skips ones whose zip is missing, so a partial download still streams whatever is on disk.

In [2]:
ds_force = RDDACDataset(view='force-curve', data_dir=DATA_DIR)
print('view:           ', ds_force.view)
print('field specs:    ', ds_force._field_specs)
print('total sim_ids:  ', len(ds_force._sim_ids), '(every experiment in process_parameters.csv)')
print('locally indexed:', len(ds_force._h5_index), '(only these will actually stream)')

view:            force-curve
field specs:     {'force_data': ('force/data', None)}
total sim_ids:   9000 (every experiment in process_parameters.csv)
locally indexed: 18 (only these will actually stream)


## 2. Single records through a `DataLoader`

`RDDACDataset` plugs straight into a `DataLoader` with no extra glue. One thing is specific to the *raw* experimental data though: the press was sampled until the operator stopped the recording, so `force/data` has a **per-experiment row count** (`n` differs). The default `collate_fn` stacks each field along a new batch axis and therefore needs equal shapes — with a ragged field that only works at `batch_size=1`.

The two experiments below make the raggedness visible: `(1, 1140, 8)` vs `(1, 1141, 8)`.

In [3]:
loader = DataLoader(RDDACDataset(view='force-curve', data_dir=DATA_DIR, sim_ids=[0, 4500]),
                    batch_size=1, num_workers=0)
for batch in loader:
    for k, v in batch.items():
        print(f'  {k:12s} shape={tuple(v.shape)} dtype={v.dtype}')

  force_data   shape=(1, 1140, 8) dtype=torch.float32
  force_data   shape=(1, 720, 8) dtype=torch.float32


## 3. Fixed-shape batching via a custom view

For real batches, give every record the same shape. The cleanest lever is a custom view that slices a **fixed window** out of the ragged table — here the first 512 rows of `force/data`, so every record is `(512, 8)`.

To stream a view built with `rddac.add_view` through `RDDACDataset`, pass the same loaded object via the `dataset=` kwarg. Without it, `RDDACDataset.__init__` would re-parse `metadata.json` from disk and the in-memory mutation would be invisible (it would raise `ValueError: view 'force-window' not found`). With it, the constructor uses the caller's object as-is.

`num_workers=2` demonstrates the automatic sharding: each worker gets a disjoint slice of the id list (`sim_ids[worker_id::num_workers]`), so no experiment is produced twice.

In [4]:
ds_manifest = rddac.load(data_dir=DATA_DIR)
rddac.add_view(
    ds_manifest,
    'force-window',
    fields={
        'window': ('force_data', list(range(512))),   # first 512 samples -> (512, 8) for every experiment
    },
)

windowed = RDDACDataset(view='force-window', data_dir=DATA_DIR, dataset=ds_manifest, sim_ids=SAMPLE_IDS)
loader = DataLoader(windowed, batch_size=4, num_workers=2)
for i, batch in enumerate(loader):
    print(f'  batch {i}: window shape={tuple(batch["window"].shape)} dtype={batch["window"].dtype}')
    if i >= 2:
        break

  batch 0: window shape=(4, 512, 8) dtype=torch.float32
  batch 1: window shape=(4, 512, 8) dtype=torch.float32


  batch 2: window shape=(4, 512, 8) dtype=torch.float32


## 4. Filter via the Croissant manifest

Both filters run against `process_parameters.csv` rows **before** any zip is opened, so the IO scales with the surviving experiments rather than with the full 9 000.

The row keys are not magic: they come straight from the Croissant manifest. `metadata.json` declares `process_parameters.csv` as a `FileObject` and exposes its columns as the `process-parameters` RecordSet. `RDDACDataset` simply consumes those rows at construction time and applies the predicate before any zip is touched.

### What `where` receives

`RDDACDataset` reads `process_parameters.csv` with `pandas` and runs `where` once per row via `df.apply(where, axis=1)`. The `row` argument is a `pandas.Series` whose index is the CSV column names, so you can read columns with either `row['split']` or `row.split`. Values come back as native Python types (strings are `str` here, not the `bytes` that `mlcroissant` yields):

| Column | Type | Example |
|--------|------|---------|
| `index` | `int` | `42` (the experiment id, matches the h5 filename) |
| `experiment_id` | `int` | `43` (1-based repetition counter within the category) |
| `category` | `int` | `0` ... `17` |
| `geometry` | `str` | `'concave'`, `'convex'` |
| `blankholder_force` | `int` | `100`, `300`, `500` (kN) |
| `mean_punch_temp` | `float` | `20.2` (degC) |
| `oil_type` | `str` | `'coarse'`, `'medium'`, `'fine'` |
| `has_pointcloud`, `has_oil` | `bool` | `True` |
| `split` | `str` | `'train'`, `'val'`, `'test'` |

Any predicate returning truthy keeps the row.

- `where=<callable>`: any function `pd.Series -> bool`.
- `sim_ids=[...]`: explicit allowlist of integers, applied before `where`.

Both can be combined; the predicate is applied after the allowlist.

In [5]:
convex = RDDACDataset(
    view='force-curve',
    data_dir=DATA_DIR,
    where=lambda row: row['geometry'] == 'convex',
)
print(f'convex-only sim_ids:        {len(convex._sim_ids):>5d} (of 9000)')

heavy_oiled = RDDACDataset(
    view='force-curve',
    data_dir=DATA_DIR,
    where=lambda row: row['blankholder_force'] == 500 and row['has_oil'],
)
print(f'500 kN + oiled sim_ids:     {len(heavy_oiled._sim_ids):>5d}')

ids_only = RDDACDataset(
    view='force-curve',
    data_dir=DATA_DIR,
    sim_ids=SAMPLE_IDS,
)
print(f'sim_ids=SAMPLE_IDS:         {len(ids_only._sim_ids):>5d}')

convex-only sim_ids:         4500 (of 9000)
500 kN + oiled sim_ids:      2988


sim_ids=SAMPLE_IDS:            18


## 5. Train / val / test splits

`process_parameters.csv` ships with a `split` column whose canonical values are `'train'`, `'val'`, and `'test'`. Because the column is part of the Croissant manifest, the same `where=` predicate that filtered by geometry above works on it. Three `RDDACDataset` instances, one per split, is the fastest way to wire up a training loop without writing any custom partitioning code.

Shuffle the train split for SGD; leave `val`/`test` deterministic for reproducible evaluation.

In [6]:
splits = {}
for name in ('train', 'val', 'test'):
    splits[name] = RDDACDataset(
        view='force-curve',
        data_dir=DATA_DIR,
        where=lambda row, n=name: row['split'] == n,
        shuffle=(name == 'train'),
        seed=42,
    )

for name, split_ds in splits.items():
    streamable = sum(1 for sid in split_ds._sim_ids if sid in split_ds._h5_index)
    print(f'{name:>5s}: {len(split_ds._sim_ids):>5d} sim_ids (of 9000), {streamable:>5d} streamable now (zip on disk)')

train:  7200 sim_ids (of 9000),    14 streamable now (zip on disk)
  val:   900 sim_ids (of 9000),     2 streamable now (zip on disk)
 test:   900 sim_ids (of 9000),     2 streamable now (zip on disk)


## 6. Shuffle + `set_epoch`

`shuffle=True` permutes each shard with a seed derived from `seed + epoch + shard_id`. Worker shards stay disjoint, so two workers do not produce the same experiment. Call `set_epoch(n)` once per epoch to get a different permutation each time. Without it, every epoch sees the same order, which biases optimisation.

Records do not carry the experiment id (that is `iter_view`'s `_sim_id` extra), so the cell below fingerprints the first record of each epoch by its data sum — the changing fingerprint shows the permutation changing.

In [7]:
ds_shuf = RDDACDataset(
    view='force-window',
    data_dir=DATA_DIR,
    dataset=ds_manifest,
    sim_ids=SAMPLE_IDS,
    shuffle=True,
    seed=42,
)
for epoch in range(3):
    ds_shuf.set_epoch(epoch)
    first = next(iter(ds_shuf))
    print(f'epoch {epoch}: first record fingerprint sum={float(first["window"].sum()):>12.1f}')

epoch 0: first record fingerprint sum=    240630.2
epoch 1: first record fingerprint sum=    232123.9
epoch 2: first record fingerprint sum=    258301.7


## 7. Metadata columns: the `RDDACDataset` limitation

Views that mix in `process-parameters` CSV columns (qualified `"process-parameters/<col>"` entries) **cannot** be streamed through `RDDACDataset`: the adapter only resolves field-map sources and raises `ValueError` at construction. Three workarounds, in order of preference:

1. **Keep the view pure h5** — the training-relevant metadata is usually better expressed as a `where=` filter or a label lookup anyway.
2. **Join the metadata yourself** — `pandas.read_csv(DATA_DIR / 'process_parameters.csv')` indexed by `index` gives you labels per experiment id; combine with `sim_ids=`.
3. **`rddac.streaming.iter_view`** — handles mixed views (it joins the CSV internally), at the cost of doing your own batching.


In [8]:
mixed = rddac.load(data_dir=DATA_DIR)
rddac.add_view(mixed, 'force-signals', fields={
    'force': 'force_data',
    'geometry': 'process-parameters/geometry',     # CSV columns -> not RDDACDataset-compatible
    'blankholder_force': 'process-parameters/blankholder_force',
    'split': 'process-parameters/split',
})

try:
    RDDACDataset(view='force-signals', data_dir=DATA_DIR, dataset=mixed)
except ValueError as e:
    print(f'RDDACDataset: ValueError: {e}')

# Workaround 3: iter_view streams the mixed view without complaint.
for rec in rddac.streaming.iter_view('force-signals', data_dir=DATA_DIR, dataset=mixed, sim_ids=[0]):
    print(f"iter_view:    force.shape={rec['force'].shape}  geometry={rec['geometry']!r}  "
          f"blankholder_force={rec['blankholder_force']} kN  split={rec['split']!r}")

RDDACDataset: ValueError: view field 'geometry' sources RecordSet 'process-parameters', but this dataset only streams 'field-map' (HDF5) fields. For views that include process-parameters metadata columns use streaming.iter_view, or build the view from field-map fields only.
iter_view:    force.shape=(1140, 8)  geometry='concave'  blankholder_force=100 kN  split='val'


## 8. Training-loop skeleton

Everything above composes into the usual PyTorch loop. The toy task: predict the **peak total press force** from the first 512 samples of the four load cells — input `(512, 4)`, target a scalar, both cut from the same `force-window` view so no metadata join is needed. This is a skeleton, not an experiment: 18 experiments, no validation, three epochs, and the raw kN scale is left unnormalised — expect the loss to be large and noisy. The point is the wiring:

- `set_epoch(epoch)` before each pass so the shuffle reseeds,
- fields arrive as a dict of tensors; slice columns inside the step,
- swap `SAMPLE_IDS` for `where=lambda r: r['split'] == 'train'` to scale the same loop to the full release.

In [9]:
import torch
import torch.nn as nn

train_ds = RDDACDataset(view='force-window', data_dir=DATA_DIR, dataset=ds_manifest,
                        sim_ids=SAMPLE_IDS, shuffle=True, seed=0)
train_loader = DataLoader(train_ds, batch_size=4, num_workers=0)

model = nn.Sequential(nn.Flatten(), nn.Linear(512 * 4, 64), nn.ReLU(), nn.Linear(64, 1))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(3):
    train_ds.set_epoch(epoch)
    total, batches = 0.0, 0
    for batch in train_loader:
        window = batch['window']                      # (B, 512, 8) float32
        x = window[:, :, 1:5]                         # the four load cells   -> (B, 512, 4)
        y = window[:, :, 7].amax(dim=1, keepdim=True) # peak total force [kN] -> (B, 1)
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
        total, batches = total + loss.item(), batches + 1
    print(f'epoch {epoch}: mean MSE loss = {total / batches:.1f} (raw kN^2 scale, illustrative only)')

epoch 0: mean MSE loss = 103331.1 (raw kN^2 scale, illustrative only)


epoch 1: mean MSE loss = 33068.9 (raw kN^2 scale, illustrative only)


epoch 2: mean MSE loss = 11153.1 (raw kN^2 scale, illustrative only)
